# Single-session optic-flow analysis

A linear, per-session walk-through of the eye/face optic-flow pipeline: **load a frame → define the
ROIs → remove the corneal reflection → run per-ROI flow tracking + pupil tracking → inspect**. Each
step drives the reusable modules in `common/`, so this notebook is the *front end*,
not a reimplementation.

**Pipeline order** (what reads what):

| step | module | writes |
|---|---|---|
| 0  load a frame | — | — |
| 1  define ROIs | you, on the frame | `roi_config_<mouse>.json` |
| 2  remove the IR glint | `common/remove_reflection.py` | `<mouse>_..._noreflection.mp4` |
| 3  build the session contract | `common/session_config.py` | `session.json` |
| 4  per-ROI optic flow | `common/compute_roi_flow.py` | `opticflow/opticflow_<roi>_<metric>.npy` |
| 5  pupil tracking | `common/detectors/segment_pupil.py` | `opticflow/pupil_track.npz` |

Two videos are kept and stay frame-aligned: the **original** (glint intact — the pupil fit anchors on
the glint) and the **noreflection** copy (glint inpainted — the flow reads this). The session folder
also needs the game **`log.json`** for `build_session`.

> **`RUN_FULL`** (set in the config cell) guards every step that *writes into the session* or decodes
> the whole video. Left `False`, a "Run All" only does the cheap previews and overwrites nothing — flip
> it to `True` to run the real, full-session pipeline.

In [ ]:
# ── imports + make ../common importable ───────────────────────────────────────
import sys, json, time
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

_HERE = Path.cwd()
# this notebook lives in common/ (next to the modules); fall back to ../common if run from elsewhere
_COMMON = _HERE if (_HERE / 'compute_roi_flow.py').exists() else (_HERE.parent / 'common')
assert (_COMMON / 'compute_roi_flow.py').exists(), f'cannot locate common/ from {_HERE}'
sys.path.insert(0, str(_COMMON))
sys.path.insert(0, str(_COMMON / 'detectors'))

import remove_reflection as rr
import compute_roi_flow as rflow
import segment_pupil as sp
import session_config as scfg
import fps as fpsmod
print('modules loaded from', _COMMON)

In [ ]:
# ── CONFIG: point at ONE session folder ───────────────────────────────────────
# The folder holds the ORIGINAL eye video (glint intact) + log.json. Everything else is produced here.
SESSION_DIR = Path('/home/maryam/repo/flow_test/JPAS_0168')   # <-- set me
MOUSE_ID    = 'JPAS_0168'                                       # <-- set me

# The safety switch. False = cheap previews only, nothing written. True = the real full-session run.
RUN_FULL = False

# window used by the PREVIEW cells (cheap: a few hundred frames)
PREVIEW_LO, PREVIEW_HI = 5000, 5200
PREVIEW_FRAME = 5000            # single frame for the ROI / reflection previews

SESSION_DIR = SESSION_DIR.resolve()
assert SESSION_DIR.exists(), SESSION_DIR
# the ORIGINAL (with-reflection) eye video = the .mp4 that is not noreflection/ui/viz
_orig = sorted(p for p in SESSION_DIR.glob('*.mp4')
               if not any(t in p.name.lower() for t in ('noreflection', 'ui', 'viz')))
ORIGINAL_VIDEO = _orig[0] if _orig else None
print('session   :', SESSION_DIR)
print('mouse     :', MOUSE_ID)
print('original  :', ORIGINAL_VIDEO)
print('RUN_FULL  :', RUN_FULL, '(False = previews only, writes nothing)')

## 0 — Load a frame

Grab one representative frame from the original video and look at it. Frame indices are the same in the
original and the (later) noreflection copy, so any index you pick here is valid everywhere.

In [ ]:
cap = cv2.VideoCapture(str(ORIGINAL_VIDEO))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
NFRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); VFPS = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, PREVIEW_FRAME)
ok, frame = cap.read(); cap.release()
assert ok, f'could not read frame {PREVIEW_FRAME}'
print(f'{W}x{H}px  {VFPS:.2f} fps  {NFRAMES} frames  ({NFRAMES/VFPS/60:.1f} min)')

plt.figure(figsize=(11, 6.2))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title(f'{MOUSE_ID}  frame {PREVIEW_FRAME}'); plt.axis('off'); plt.show()

## 1 — Define the ROIs (stepwise)

An ROI is a box `[x1, y1, x2, y2]` in **original-video pixels**. We do this in small steps so you never
have to scroll: **1a** shows the frame under a numbered grid to read coordinates off; **1b** loads the
helpers; **1c** you add ROIs one at a time — each `set_roi(...)` **redraws the frame right below it**, so
you tweak the numbers and re-run *that one cell*; **1d** zooms a single ROI to fine-tune its edges;
**1e** saves. The eye boxes (`left_eye`, `left_fovea`) matter twice — their glint is what step 2
inpaints, and `left_fovea` gates the pupil fit in step 5.

**1a — the coordinate grid.** Major gridlines + tick labels every `GRID_STEP` px, faint minor lines at
the half-step. Read the corners of the box you want off the axes, then type them into `set_roi` below.

In [ ]:
GRID_STEP = 100   # px between labelled gridlines -- lower it (e.g. 50) to read finer

def show_grid(rois=None, step=GRID_STEP, region=None, title=None, figsize=(13, 7.6)):
    '''Show the frame with a pixel ruler + grid, and (optionally) the ROIs drawn on top.
       region=(x1,y1,x2,y2) zooms in; rois=dict draws boxes+labels.'''
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for name, r in (rois or {}).items():
        x1, y1, x2, y2 = r['bbox']; col = [c / 255 for c in r['color_bgr'][::-1]]  # BGR->RGB
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=col, lw=2))
        ax.text(x1 + 2, y1 - 4, name, color=col, fontsize=9, fontweight='bold')
    ax.set_xticks(np.arange(0, W + 1, step)); ax.set_yticks(np.arange(0, H + 1, step))
    ax.set_xticks(np.arange(0, W + 1, max(10, step // 2)), minor=True)
    ax.set_yticks(np.arange(0, H + 1, max(10, step // 2)), minor=True)
    ax.grid(which='major', color='#ffd000', alpha=0.55, lw=0.8)
    ax.grid(which='minor', color='#ffd000', alpha=0.20, lw=0.5)
    ax.tick_params(labelsize=8)
    if region:
        rx1, ry1, rx2, ry2 = region; ax.set_xlim(rx1, rx2); ax.set_ylim(ry2, ry1)  # y inverted
    ax.set_title(title or f'{MOUSE_ID}  frame {PREVIEW_FRAME} — read x,y off the grid (px)')
    plt.tight_layout(); plt.show()

show_grid()

**1b — load the helpers.** `set_roi(name, x1, y1, x2, y2)` adds/updates one box and redraws;
`drop_roi(name)` removes one; `START_FROM_EXISTING` decides whether you begin from this session's saved
`roi_config` (nudge a few) or from an empty board (define from scratch).

In [ ]:
START_FROM_EXISTING = True      # False -> start with NO ROIs and build them up yourself

_cfg = sorted(SESSION_DIR.glob('roi_config*.json'))
if START_FROM_EXISTING and _cfg:
    ROIS = {k: {'bbox': list(v['bbox']), 'color_bgr': list(v.get('color_bgr', [0, 255, 0]))}
            for k, v in json.load(open(_cfg[0])).items()}
    print('loaded', _cfg[0].name, '->', list(ROIS))
else:
    ROIS = {}
    print('starting EMPTY -- add ROIs with set_roi(...) below')

_PALETTE = [(0,255,0), (0,0,255), (255,0,255), (0,128,255), (255,165,0), (0,255,255), (255,255,0)]

def set_roi(name, x1, y1, x2, y2, color=None):
    '''Add/replace ONE ROI and immediately redraw the whole set on the grid, right here.'''
    x1, x2 = sorted((int(x1), int(x2))); y1, y2 = sorted((int(y1), int(y2)))
    color = color or ROIS.get(name, {}).get('color_bgr') or _PALETTE[len(ROIS) % len(_PALETTE)]
    ROIS[name] = {'bbox': [x1, y1, x2, y2], 'color_bgr': list(color)}
    show_grid(rois=ROIS, title=f'set {name} = [{x1}, {y1}, {x2}, {y2}]   ({len(ROIS)} ROIs total)')

def drop_roi(name):
    ROIS.pop(name, None); show_grid(rois=ROIS, title=f'dropped {name}   ({len(ROIS)} ROIs total)')

print('helpers ready:  set_roi(name, x1,y1,x2,y2) · drop_roi(name)')

**1c — add one ROI at a time.** Edit the numbers and re-run this cell; the frame under it updates with
every ROI drawn. Keep a line per ROI (this session uses `whisker_left/right`, `nose`, `mouth`, `paw`,
`left_eye`, `left_fovea`). The template below defines the two **eye** boxes step 2 needs — extend it.

In [ ]:
# one line per ROI -- read the corners off the 1a grid, edit, re-run. Comment out what you don't need.
set_roi('left_eye',   615, 168, 710, 233)
set_roi('left_fovea', 630, 175, 676, 206)
# set_roi('whisker_right', 875,  20, 1200, 220)
# set_roi('whisker_left',  425, 285,  750, 450)
# set_roi('nose',          840, 245,  925, 330)
# set_roi('mouth',         760, 333,  910, 420)
# set_roi('paw',           360, 500,  740, 680)

**1d — zoom to fine-tune one box.** Pass a ROI name to see it enlarged on a fine grid, so you can read
the exact edges and correct them with another `set_roi` in 1c.

In [ ]:
def zoom_roi(name, pad=40, step=20):
    x1, y1, x2, y2 = ROIS[name]['bbox']
    show_grid(rois={name: ROIS[name]}, step=step, figsize=(9, 7),
              region=(max(0, x1 - pad), max(0, y1 - pad), min(W, x2 + pad), min(H, y2 + pad)),
              title=f'{name} = [{x1},{y1},{x2},{y2}]  (grid {step}px) — fine-tune the edges')

zoom_roi('left_eye')      # <-- change the name to inspect any ROI

**1e — save.** Writes `roi_config_<mouse>.json` (guarded by `RUN_FULL`), which every downstream step
reads.

*Prefer to drag boxes instead of typing coordinates? If your Jupyter has `ipympl`, run `%matplotlib
widget` in a cell, then use `matplotlib.widgets.RectangleSelector` (its `onselect` gives you the
`x1,y1,x2,y2` to paste into `set_roi`). On a local display, `cv2.selectROI("roi", frame)` returns the
same four numbers. Both just feed `set_roi` — the grid workflow above needs neither.*

In [ ]:
# SAVE the ROI config (guarded). Downstream (build_session, remove_reflection) reads this file.
roi_path = SESSION_DIR / f'roi_config_{MOUSE_ID}.json'
if RUN_FULL:
    json.dump({n: {'bbox': [int(v) for v in r['bbox']],
                   'color_bgr': [int(c) for c in r['color_bgr']]} for n, r in ROIS.items()},
              open(roi_path, 'w'), indent=2)
    print('wrote', roi_path)
else:
    print('RUN_FULL is False -> not writing', roi_path.name, '(flip RUN_FULL to save)')

## 2 — Remove the corneal reflection (IR glint)

The IR LEDs leave a near-white specular glint on the cornea. It sits over the pupil and confuses both
the flow and the pupil fit, so it is inpainted out of the **eye boxes only** — producing the
`*_noreflection.mp4` that the flow step reads. First preview the mask + inpaint on one frame (adjust
`threshold` if the mask misses the glint or eats too much), then write the full video.

In [ ]:
# preview the glint mask + inpaint on the eye crop (cheap, one frame)
eye_names = [n for n in ('left_eye', 'left_fovea', 'eye', 'fovea') if n in ROIS]
assert eye_names, 'need an eye/fovea ROI for reflection removal'
eb = ROIS[eye_names[0]]['bbox']
crop, mask, cleaned = rr.preview(ORIGINAL_VIDEO, eb, frame=PREVIEW_FRAME, threshold=rr.THRESHOLD)
print(f'glint pixels >{rr.THRESHOLD}: {(crop>rr.THRESHOLD).sum()} | mask px: {(mask>0).sum()} | '
      f'max in crop {crop.max()} -> after inpaint {cleaned.max()}')
z = 6; sz = (crop.shape[1]*z, crop.shape[0]*z)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, im, t in zip(ax, [crop, mask, cleaned], ['original eye crop', 'glint mask', 'inpainted']):
    a.imshow(cv2.resize(im, sz, interpolation=cv2.INTER_NEAREST), cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# WRITE the FULL noreflection video -- the WHOLE session, every frame (hi=None). Guarded because it
# decodes + inpaints + re-encodes the entire video (minutes); the one-frame preview above is the cheap
# check, this is the real thing.
noref_path = SESSION_DIR / f'{MOUSE_ID}_noreflection.mp4'
eye_bboxes = [ROIS[n]['bbox'] for n in eye_names]
if RUN_FULL:
    if noref_path.exists():
        print('already exists ->', noref_path.name, '  (delete it to regenerate)')
    else:
        rr.generate_noreflection_video(ORIGINAL_VIDEO, noref_path, eye_bboxes,   # hi=None => WHOLE video
                                       threshold=rr.THRESHOLD, dilate_iter=rr.DILATE_ITER,
                                       inpaint_radius=rr.INPAINT_RADIUS)
else:
    print('RUN_FULL is False -> NOT generating the noreflection video (nothing written).')
    print('   set RUN_FULL = True to inpaint the WHOLE session ->', noref_path.name)

## 3 — Build the session contract (`session.json`)

`build_session` discovers the two videos + the ROI config + `log.json`, derives fps from the log
camera table and cross-checks the video, classifies the task, runs the camera-shift check, and writes
**`session.json`** — the single file every downstream step reads. It **merges** into any existing
`session.json`, so manual keys (e.g. `view_scale`) survive a rebuild.

In [ ]:
if RUN_FULL:
    sess = scfg.build_session(MOUSE_ID, str(SESSION_DIR), write=True, verbose=True)
else:
    _sp = SESSION_DIR / 'session.json'
    if _sp.exists():
        sess = json.load(open(_sp)); print('RUN_FULL is False -> loaded existing session.json')
    else:
        sess = scfg.build_session(MOUSE_ID, str(SESSION_DIR), write=False, verbose=True)
        print('(built in-memory; not written because RUN_FULL is False)')
print('\ntask:', sess.get('task_type'), '| fps:', round(sess.get('fps', 0), 3),
      '| frames:', sess.get('n_frames'), '| camera_moved:', sess.get('camera_moved'))
print('rois:', list(sess.get('rois', {})))

## 4 — Per-ROI optic flow

`compute_roi_flow` runs Farneback per ROI (on the 0.5-scaled noreflection frame, padded then sliced)
and averages the 8 flow metrics — `mag, x, y, angle, coherence, variance, divergence, radial` — into
`opticflow/opticflow_<roi>_<metric>.npy` (one value per frame). Preview a short window first (returns
the arrays without writing), then run the whole session.

In [ ]:
# PREVIEW: a few hundred frames, nothing written -- returns {roi: {metric: array}}
arr = rflow.run(str(SESSION_DIR), lo=PREVIEW_LO, hi=PREVIEW_HI, write=False, progress=False)

t = np.arange(PREVIEW_LO, PREVIEW_HI)
fig, ax = plt.subplots(figsize=(12, 4))
for roi in [r for r in ('paw', 'whisker_left', 'whisker_right', 'mouth', 'nose') if r in arr]:
    ax.plot(t, arr[roi]['mag'][PREVIEW_LO:PREVIEW_HI], lw=1.1, label=roi)
ax.set_xlabel('frame'); ax.set_ylabel('|flow| (mag)')
ax.set_title(f'per-ROI motion energy, frames {PREVIEW_LO}-{PREVIEW_HI}'); ax.legend(fontsize=8); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

In [ ]:
# FULL RUN (guarded -- ~30 min for a full session; writes opticflow_<roi>_<metric>.npy)
if RUN_FULL:
    rflow.run(str(SESSION_DIR), write=True)
else:
    print('RUN_FULL is False -> skipping the full flow write (the preview above is unwritten)')

## 5 — Pupil tracking

`segment_pupil` fits the dark iris disc anchored on the corneal glint, on the **original** (glint-intact)
video, gated to `left_fovea`. **QC first**: `overlay_grid` renders a contact sheet of the fit on sample
frames — *look at it* before trusting the numbers (this fit is per-animal and only approximate for
pupil size; position + blink are the reliable parts). Then run the session → `pupil_track.npz`.

In [ ]:
# QC contact sheet -- writes a PNG to debug/, always safe to run
qc_frames = list(range(PREVIEW_LO, PREVIEW_LO + 180, 30))
qc_png = SESSION_DIR / 'debug' / 'pupil_overlay_qc.png'
sp.overlay_grid(str(SESSION_DIR), qc_frames, out_path=str(qc_png))
plt.figure(figsize=(13, 5)); plt.imshow(plt.imread(qc_png)); plt.axis('off')
plt.title('pupil fit QC -- eyeball this before trusting radius'); plt.show()

In [ ]:
# FULL RUN (guarded -- decodes the video; writes opticflow/pupil_track.npz)
if RUN_FULL:
    res = sp.run(str(SESSION_DIR), write=True)
    r = res['radius']
else:
    _pt = SESSION_DIR / 'opticflow' / 'pupil_track.npz'
    if _pt.exists():
        r = np.load(_pt)['radius']; print('RUN_FULL is False -> loaded existing pupil_track.npz')
    else:
        res = sp.run(str(SESSION_DIR), lo=PREVIEW_LO, hi=PREVIEW_HI, write=False, progress=False)
        r = res['radius']; print('(fit a short window in-memory; not written)')
print(f'radius: valid {100*np.mean(~np.isnan(r)):.1f}%  median {np.nanmedian(r):.1f} px')

## 6 — Inspect the outputs

Everything downstream (grooming, licking, whisker, eye events, the trial report) reads these arrays.
A quick look: per-ROI motion energy + the pupil radius over the preview window.

In [ ]:
ofdir = SESSION_DIR / 'opticflow'
def load_metric(roi, metric):
    p = ofdir / f'opticflow_{roi}_{metric}.npy'
    return np.load(p) if p.exists() else None

fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for roi in ('paw', 'whisker_left', 'whisker_right', 'mouth'):
    m = load_metric(roi, 'mag')
    src = m if m is not None else (arr[roi]['mag'] if roi in arr else None)
    if src is not None:
        ax[0].plot(np.arange(PREVIEW_LO, PREVIEW_HI), src[PREVIEW_LO:PREVIEW_HI], lw=1.1, label=roi)
ax[0].set_ylabel('|flow|'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.2)
ax[0].set_title(f'{MOUSE_ID}  outputs over frames {PREVIEW_LO}-{PREVIEW_HI}')
ax[1].plot(np.arange(PREVIEW_LO, PREVIEW_HI), r[PREVIEW_LO:PREVIEW_HI], color='#8e44ad', lw=1.2)
ax[1].set_ylabel('pupil radius (px)'); ax[1].set_xlabel('frame'); ax[1].grid(alpha=.2)
plt.tight_layout(); plt.show()

print('optic-flow .npy files:', len(sorted(ofdir.glob('opticflow_*_*.npy'))) if ofdir.exists() else 0)
print('\\nNext, per-animal detectors run on these arrays (see ../common/detectors/):')
print('  compute_eye_events · detect_grooming · detect_licking · compute_mouth_state · detect_saccades')